# UdaPlay AI Research Agent

## Part 2 - Agent Implementation

This notebook implements the agent layer for UdaPlay, an AI research assistant for the video game industry.

UdaPlay combines local retrieval with web search to answer questions about games, publishers, developers, platforms, release dates, and current industry activity.

The agent will:

1. Retrieve relevant information from the local ChromaDB knowledge base
2. Evaluate whether the retrieved information is sufficient and reliable
3. Fall back to web search when local retrieval is insufficient
4. Maintain conversation state across multiple queries
5. Coordinate tool usage through a state-machine workflow
6. Return clear, structured answers with source information

### Setup

In [ ]:
# Compatibility setup for the Udacity workspace
import importlib.util
import sys

if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules["sqlite3"] = sys.modules.pop("pysqlite3")

In [ ]:
import os
import json
import chromadb

from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from tavily import TavilyClient
from pydantic import BaseModel, Field

from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.parsers import PydanticOutputParser

In [ ]:
# Load environment variables
load_dotenv()

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("CHROMA_OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

### Tools

#### Retrieve Game Tool

In [ ]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("CHROMA_OPENAI_API_KEY"),
    api_base=os.getenv("OPENAI_BASE_URL"),
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path="chromadb")

collection = chroma_client.get_collection(
    name="udaplay",
    embedding_function=embedding_fn
)


@tool
def retrieve_game(query: str) -> list:
    """
    Semantic search for relevant game information in the local vector database.

    Args:
        query: A question about the video game industry.

    Returns:
        A list of matching game records including their local source file.
    """

    results = collection.query(
        query_texts=[query],
        n_results=3
    )

    games = []

    for doc_id, metadata in zip(
        results["ids"][0],
        results["metadatas"][0]
    ):
        game = dict(metadata)
        game["source"] = f"games/{doc_id}.json"
        games.append(game)

    return games

#### Evaluate Retrieval Tool

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(
        description="Whether the retrieved documents are sufficient to answer the question"
    )
    description: str = Field(
        description="Explanation of why the retrieved documents are or are not useful"
    )


@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """
    Evaluate whether retrieved documents are sufficient to answer the user's question.

    Args:
        question: The original user question.
        retrieved_docs: Documents retrieved from the local vector database.

    Returns:
        An evaluation containing:
        - useful: whether the documents can answer the question
        - description: explanation of the evaluation
    """

    llm_judge = LLM(model="gpt-4o-mini")

    prompt = f"""
    Your task is to evaluate whether the retrieved documents contain
    enough information to accurately answer the user's question.

    User question:
    {question}

    Retrieved documents:
    {json.dumps(retrieved_docs, indent=2)}

    Determine whether the documents are useful enough to answer the question.
    Give a clear explanation for your decision.
    """

    response = llm_judge.invoke(
        input=prompt,
        response_format=EvaluationReport
    )

    parser = PydanticOutputParser(model_class=EvaluationReport)
    evaluation = parser.parse(response)

    return evaluation.model_dump()

#### Game Web Search Tool

In [ ]:
tavily_client = TavilyClient(
    api_key=os.getenv("TAVILY_API_KEY")
)


@tool
def game_web_search(question: str) -> list:
    """
    Search the web for relevant video game information when the local
    vector database does not contain enough information.

    Args:
        question: A question about the video game industry.

    Returns:
        A list of web search results containing title, URL, and content.
    """

    response = tavily_client.search(
        query=question,
        search_depth="advanced",
        max_results=5
    )

    results = []

    for result in response.get("results", []):
        results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        })

    return results

### Agent

In [ ]:
udaplay_agent = Agent(
    model_name="gpt-4o-mini",
    temperature=0.2,
    instructions="""
You are UdaPlay, an AI research assistant for the video game industry.

Your job is to answer questions about video games, including:
- game titles
- developers and publishers
- release dates
- platforms
- genres
- descriptions
- current company or game activity

Follow this workflow:

1. Always begin by calling retrieve_game with the user's question.
2. After retrieving local results, call evaluate_retrieval using:
   - the original user question
   - the retrieved documents
3. If evaluate_retrieval returns useful=True:
   - answer using the retrieved information
   - do not perform a web search unless the user explicitly asks for current information
4. If evaluate_retrieval returns useful=False, or the retrieved information is incomplete:
   - call game_web_search using the user's question
5. For questions about current or recent activity, prefer web search if the local dataset does not contain current information.
6. Do not invent facts. Base answers only on retrieved documents or web search results.
7. Give a concise, readable final answer.
8. Cite the source used:
   - For local results, cite the game JSON source returned by retrieve_game.
   - For web results, include the relevant source URL.
9. Clearly indicate whether the answer came from the local game database, web search, or both.
""",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search
    ]
)

### Agent Demonstration

The following examples demonstrate local retrieval, retrieval evaluation, web-search fallback, tool usage, and conversation state within a shared session.

In [ ]:
questions = [
    "When were Pokémon Gold and Silver released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

session_id = "udaplay_demo"

for i, question in enumerate(questions, start=1):

    run = udaplay_agent.invoke(
        question,
        session_id=session_id
    )

    final_state = run.get_final_state()
    messages = final_state["messages"]

    # Find the most recent user message so only the current query's
    # tool usage and final answer are displayed.
    start_index = 0

    for index in range(len(messages) - 1, -1, -1):
        if isinstance(messages[index], UserMessage):
            start_index = index
            break

    current_messages = messages[start_index:]

    print("\n" + "=" * 70)
    print(f"Question {i}: {question}")
    print("=" * 70)

    print("\nTool Usage:")

    for message in current_messages:
        if isinstance(message, AIMessage) and message.tool_calls:
            for tool_call in message.tool_calls:
                print(f"- {tool_call.function.name}")

    final_answer = next(
        message.content
        for message in reversed(current_messages)
        if isinstance(message, AIMessage) and message.content
    )

    print("\nFinal Answer:")
    print(final_answer)